We extract features from time-series using Fourier series and Splines.

```python
FourierCoefficients = FourierCoefficients(homogeneous_timeseries, order=10, eps=1e-6)
DiffCollection = DiffCollection(homogeneous_timeseries, n=5, k=3)
```


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy import interpolate

from numba import njit

In [9]:
def FourierCoefficientsClaude(timeseries, number_of_components):
    N = len(timeseries)
    t = np.arange(N)
    frequencies = np.fft.fftfreq(N)[:number_of_components]
    
    coefficients = []
    for freq in frequencies:
        cos_comp = np.sum(timeseries * np.cos(2 * np.pi * freq * t)) * 2 / N
        sin_comp = np.sum(timeseries * np.sin(2 * np.pi * freq * t)) * 2 / N
        coefficients.append((freq, cos_comp, sin_comp))
    return coefficients

def ReconstructTimeSeriesClaude(FourierComponents, Length_of_timeseries, MeanValue):
    t = np.arange(Length_of_timeseries)
    reconstructed = np.zeros(Length_of_timeseries) + MeanValue
    
    for freq, cos_comp, sin_comp in FourierComponents:
        reconstructed += cos_comp * np.cos(2 * np.pi * freq * t)
        reconstructed += sin_comp * np.sin(2 * np.pi * freq * t)
    return reconstructed

def FourierCoefficientsGPT(timeseries, number_of_components):
    """
    Calculate the Fourier coefficients for a given time series.

    Parameters:
    timeseries (array-like): The input time series data.
    number_of_components (int): The number of Fourier components to calculate.

    Returns:
    np.ndarray: Fourier coefficients (complex numbers).
    """
    N = len(timeseries)
    # Compute the Fourier Transform
    fft_result = np.fft.fft(timeseries)
    # Normalize and select the first 'number_of_components' coefficients
    coefficients = fft_result[:number_of_components] / N
    return coefficients

def ReconstructTimeSeriesGPT(FourierComponents, Length_of_timeseries, MeanValue):
    """
    Reconstruct a time series from its Fourier components.

    Parameters:
    FourierComponents (array-like): The Fourier coefficients (complex numbers).
    Length_of_timeseries (int): The length of the original time series.
    MeanValue (float): The mean value of the original time series.

    Returns:
    np.ndarray: Reconstructed time series.
    """
    N = Length_of_timeseries
    # Initialize the reconstructed series
    reconstructed_series = np.zeros(N)
    
    # Reconstruct the time series
    for k in range(len(FourierComponents)):
        amplitude = np.abs(FourierComponents[k])
        phase = np.angle(FourierComponents[k])
        reconstructed_series += amplitude * np.cos(2 * np.pi * k * np.arange(N) / N + phase)
    
    # Add the mean value
    reconstructed_series += MeanValue

    return reconstructed_series


def performance_test(n_samples=100_000, n_components=50, n_tests=10):
    print(f"Running performance test with {n_samples} samples, {n_components} components, {n_tests} times.")
    
    total_time_fourier = 0
    total_time_reconstruct = 0
    _corr = 0
    for _ in range(n_tests):
        # Generate a sample time series
        t = np.linspace(0, 10, n_samples)
        timeseries = np.sin(2*np.pi*t) + 0.5*np.sin(4*np.pi*t) + np.random.normal(0, 0.1, n_samples)
        
        # Time FourierCoefficients
        start = time.time()
        coeffs = FourierCoefficientsGPT(timeseries, n_components)
        end = time.time()
        total_time_fourier += end - start
        
        # Time ReconstructTimeSeries
        start = time.time()
        reconstructed = ReconstructTimeSeriesGPT(coeffs, n_samples, np.mean(timeseries))
        end = time.time()
        total_time_reconstruct += end - start
        
        _corr += np.corrcoef(timeseries, reconstructed)[0,1]
    
    avg_time_fourier = total_time_fourier / n_tests
    avg_time_reconstruct = total_time_reconstruct / n_tests
    mcorr = _corr / n_tests
    
    print(f"Average time for FourierCoefficients: {avg_time_fourier:.6f} seconds")
    print(f"Average time for ReconstructTimeSeries: {avg_time_reconstruct:.6f} seconds")
    print(f"Average correlation of Reconstruction: {mcorr:.6f}")


In [10]:
ts = np.array([1,2,3,4,8,6,7,8,9,10])
ts2 = np.diff(ts)
np.diff(ts2)

array([ 0,  0,  3, -6,  3,  0,  0,  0])

In [11]:
@njit
def DiffCollector(ts: np.ndarray, n: int=3, k: int=5):
    '''
        Function to get n-regularly spaced kth order discrete differences
        
        ts: homogeneous time series  
        k: number of points in regular intervals
        n: max order of the discrete differences
        
        return: [n-regularly spaced differences * order of the differences]
    '''
    N = len(ts)
    res = []
    diffs = ts
    for _n in range(n):
        ts_step = (N-_n) // k       
        diffs = np.diff(diffs)
        res.extend(diffs[np.arange(0, len(diffs), ts_step)])
    return res

In [17]:
performance_test(n_samples=10_000, n_components=25, n_tests=10)

Running performance test with 10000 samples, 25 components, 10 times.
Average time for FourierCoefficients: 0.000296 seconds
Average time for ReconstructTimeSeries: 0.003870 seconds
Average correlation of Reconstruction: 0.992164


In [13]:
x = np.linspace(0, 10, 1000)
y = 2*np.sin(3*x)

coeffs = DiffCollector(y, n=3, k=2)

print(f"Diffs")
for i, coeff in enumerate(coeffs):
    print(coeff)

Diffs
0.060051033412992276
-0.046777258671510724
-5.415011470927267e-05
-0.001152071414952438
-5.4101285659087506e-05
4.218068830552646e-05
